In [1]:
from ultralytics import YOLO
import cv2
import time
from collections import defaultdict
import os
import shutil

<h1>Prepare the WIDER Face Dataset for YOLO model</h1>

In [7]:
def convert_widerface_to_yolo(wider_root, output_root, split):
    img_src = os.path.join(wider_root, f'WIDER_{split}/images')
    label_txt = os.path.join(wider_root, 'wider_face_split', f'wider_face_{split}_bbx_gt.txt')
    img_dst = os.path.join(output_root, split, 'images')
    lbl_dst = os.path.join(output_root, split, 'labels')

    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)

    with open(label_txt, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]  # skip blank lines

    idx = 0
    while idx < len(lines):
        img_rel_path = lines[idx]
        try:
            num_boxes = int(lines[idx + 1])
        except ValueError:
            print(f"Skipping invalid entry at line {idx}: {lines[idx]}")
            idx += 1
            continue

        img_path = os.path.join(img_src, img_rel_path)
        img_out_path = os.path.join(img_dst, img_rel_path)
        label_out_path = os.path.join(lbl_dst, img_rel_path.replace('.jpg', '.txt'))

        os.makedirs(os.path.dirname(img_out_path), exist_ok=True)
        os.makedirs(os.path.dirname(label_out_path), exist_ok=True)

        try:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Image not found: {img_path}")
                idx += 2 + num_boxes
                continue
            ih, iw = img.shape[:2]
        except:
            print(f"Error loading image: {img_path}")
            idx += 2 + num_boxes
            continue

        shutil.copy(img_path, img_out_path)

        with open(label_out_path, 'w') as label_file:
            for i in range(num_boxes):
                try:
                    box_info = list(map(float, lines[idx + 2 + i].split()))
                    x, y, w, h = box_info[:4]
                    x_center = (x + w / 2) / iw
                    y_center = (y + h / 2) / ih
                    w_norm = w / iw
                    h_norm = h / ih
                    label_file.write(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                except Exception as e:
                    print(f"Error parsing box at line {idx + 2 + i}: {lines[idx + 2 + i]}")

        idx += 2 + num_boxes
wider_root = '/home/saku/Desktop/Lab/Assignment_7/datasets/WIDERFACE'
output_root = '/home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO'

convert_widerface_to_yolo(wider_root, output_root, 'train')
convert_widerface_to_yolo(wider_root, output_root, 'val')


Skipping invalid entry at line 10423: 0 0 0 0 0 0 0 0 0 0
Skipping invalid entry at line 86538: 0 0 0 0 0 0 0 0 0 0
Skipping invalid entry at line 133393: 0 0 0 0 0 0 0 0 0 0
Skipping invalid entry at line 145713: 0 0 0 0 0 0 0 0 0 0


In [8]:
import os
print(len(os.listdir('/home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/images')))
print(len(os.listdir('/home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/labels')))


61
62


<h2>Fine-tune any Ultralytics YOLOv8 model to prepare it as a face detector using WIDER FACE training set by splitting it into training set and validation set.</h2>

<h1>Model train using YOLO v8</h1>

In [51]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='widerface.yaml', epochs=30, imgsz=640, batch=8, lr0=0.005, lrf=0.01, save=True, warmup_epochs=5)

New https://pypi.org/project/ultralytics/8.3.161 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.160 🚀 Python-3.11.13 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce GTX 1070, 8104MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=widerface.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train20, nbs=64, nms=False, opset=None, optimize=Fa

train: Scanning /home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/labels/0--Parade.cache... 12880 images, 4 backgrounds, 1 corrupt: 100%|██████████| 12880/12880 [00:00<?, ?it/s]

train: /home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/images/2--Demonstration/2_Demonstration_Protesters_2_231.jpg: 1 duplicate labels removed
train: /home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/images/37--Soccer/37_Soccer_Soccer_37_851.jpg: 1 duplicate labels removed
train: /home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/images/54--Rescue/54_Rescue_rescuepeople_54_29.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.025391]
train: /home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/train/images/7--Cheering/7_Cheering_Cheering_7_17.jpg: 1 duplicate labels removed


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1063.4±813.0 MB/s, size: 116.1 KB)


val: Scanning /home/saku/Desktop/Lab/Assignment_7/datasets/WIDER_YOLO/val/labels/0--Parade.cache... 3226 images, 0 backgrounds, 1 corrupt: 100%|██████████| 3226/3226 [00:00<?, ?it/s]

...

      30/30      5.42G      1.355     0.6527     0.9845         10        640: 100%|██████████| 1610/1610 [04:32<00:00,  5.92it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 202/202 [00:28<00:00,  7.13it/s]


                   all       3225      39675      0.839       0.56      0.644      0.351

30 epochs completed in 2.411 hours.
Optimizer stripped from runs/detect/train20/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train20/weights/best.pt, 6.2MB

Validating runs/detect/train20/weights/best.pt...
Ultralytics 8.3.160 🚀 Python-3.11.13 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce GTX 1070, 8104MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 202/202 [00:25<00:00,  7.80it/s]


                   all       3225      39675       0.84       0.56      0.644      0.351
Speed: 0.2ms preprocess, 3.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/train20


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7764d5d34810>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [23]:
def show_output(model, input_dir, output_dir):     

    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(input_dir, filename)

            results = model(image_path)

            output_image = results[0].plot()
            output_path = os.path.join(output_dir, filename)
            cv2.imwrite(output_path, output_image)

            print(f"Processed and saved: {output_path}")

In [24]:
model = YOLO('runs/detect/train16/weights/best.pt')
input_dir = 'Input_images'   
output_dir = 'Output_images'   
show_output(model, input_dir, output_dir)     



image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test.jpg: 192x256 4 faces, 7.6ms
Speed: 0.4ms preprocess, 7.6ms inference, 1.1ms postprocess per image at shape (1, 3, 192, 256)
Processed and saved: Output_images/test.jpg

image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test_3.jpg: 160x256 12 faces, 7.8ms
Speed: 0.4ms preprocess, 7.8ms inference, 1.6ms postprocess per image at shape (1, 3, 160, 256)
Processed and saved: Output_images/test_3.jpg

image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test_image.jpg: 192x256 3 faces, 7.4ms
Speed: 1.0ms preprocess, 7.4ms inference, 1.1ms postprocess per image at shape (1, 3, 192, 256)
Processed and saved: Output_images/test_image.jpg


<h3>Detect the face from the video</h3>

In [26]:
cap = cv2.VideoCapture('test_2.mp4')
version = "YOLOv8n"
calculate_inference_object(model, cap, version)

YOLOv8n result : 
Processed 1081 frames in 14.51 seconds
YOLOv8n - Average FPS (incl. display): 74.49
Average inference time per frame: 0.008 seconds

Detected objects summary:
face: 340


# Using Yusepp pretrained model

In [27]:
model = YOLO('runs/detect/train16/weights/best.pt')
input_dir = 'Input_images'   
output_dir = 'Output_images'   
show_output(model, input_dir, output_dir) 


image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test.jpg: 192x256 4 faces, 9.3ms
Speed: 0.5ms preprocess, 9.3ms inference, 1.9ms postprocess per image at shape (1, 3, 192, 256)
Processed and saved: Output_images/test.jpg

image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test_3.jpg: 160x256 12 faces, 8.4ms
Speed: 0.5ms preprocess, 8.4ms inference, 1.6ms postprocess per image at shape (1, 3, 160, 256)
Processed and saved: Output_images/test_3.jpg

image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test_image.jpg: 192x256 3 faces, 8.3ms
Speed: 0.8ms preprocess, 8.3ms inference, 1.2ms postprocess per image at shape (1, 3, 192, 256)
Processed and saved: Output_images/test_image.jpg


In [28]:
model_yolo_8_yusepp = YOLO('yolov8l_100e.pt')
input_dir = 'Input_images'   
output_dir = 'Output_images_Yusepp1' 
show_output(model_yolo_8_yusepp, input_dir, output_dir)


image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test.jpg: 448x640 4 Faces, 34.2ms
Speed: 1.3ms preprocess, 34.2ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)
Processed and saved: Output_images_Yusepp1/test.jpg

image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test_3.jpg: 384x640 12 Faces, 31.3ms
Speed: 1.2ms preprocess, 31.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: Output_images_Yusepp1/test_3.jpg

image 1/1 /home/saku/Desktop/Lab/Assignment_7/Input_images/test_image.jpg: 480x640 75 Faces, 33.6ms
Speed: 2.3ms preprocess, 33.6ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)
Processed and saved: Output_images_Yusepp1/test_image.jpg


<h1>Object detection from the captured video</h1>

<h2>Capture a video without any human face using a smart phone and detect object using Ultralytics provided YOLOv8, YOLOv11 and YOLOv12 models. Compare their object detection capability, speed.</h2>

In [14]:
def calculate_inference_object(model, cap, version):
    frame_count = 0
    total_inference_time = 0
    object_counter = defaultdict(int)

    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        inference_start = time.time()
        results = model(frame, verbose=False)
        inference_end = time.time()

        total_inference_time += (inference_end - inference_start)

        result_frame = results[0].plot()
        frame_count += 1

        for box in results[0].boxes:
            cls_id = int(box.cls[0])  
            class_name = model.names[cls_id]
            object_counter[class_name] += 1

        cv2.imshow("Detection", result_frame)
        if cv2.waitKey(1) == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    end_time = time.time()

    total_time = end_time - start_time
    fps = frame_count / total_time
    avg_inference_time = total_inference_time / frame_count

    print(f"{version} result : ")
    print(f"Processed {frame_count} frames in {total_time:.2f} seconds")
    print(f"{version} - Average FPS (incl. display): {fps:.2f}")
    print(f"Average inference time per frame: {avg_inference_time:.3f} seconds")

    print("\nDetected objects summary:")
    for obj, count in object_counter.items():
        print(f"{obj}: {count}")
    

<h1>YOLOv8n</h1>

In [15]:
model_yolo_8 = YOLO('yolov8n.pt')
cap = cv2.VideoCapture('test_2.mp4')
version = "YOLOv8n"
calculate_inference_object(model_yolo_8, cap, version)

YOLOv8n result : 
Processed 1081 frames in 17.47 seconds
YOLOv8n - Average FPS (incl. display): 61.89
Average inference time per frame: 0.010 seconds

Detected objects summary:
bird: 17
cake: 11
motorcycle: 11
person: 2642
car: 85
clock: 8
boat: 16
handbag: 44
tie: 2
snowboard: 4
bottle: 4
skateboard: 7
carrot: 4
airplane: 67
bed: 1
cell phone: 8
suitcase: 4
umbrella: 14
bus: 37
train: 15


<h2>YOLO11n</h2>

In [16]:
model_yolo_11 = YOLO('yolo11n.pt')
cap = cv2.VideoCapture('test_2.mp4')
version = "YOLO11n"
calculate_inference_object(model_yolo_11, cap, version)

YOLO11n result : 
Processed 1081 frames in 19.36 seconds
YOLO11n - Average FPS (incl. display): 55.83
Average inference time per frame: 0.011 seconds

Detected objects summary:
pizza: 2
bird: 11
person: 2444
motorcycle: 35
car: 149
airplane: 110
umbrella: 30
handbag: 36
sports ball: 4
cell phone: 24
suitcase: 4
backpack: 18
carrot: 17
train: 67
parking meter: 8
boat: 9
bus: 6
hot dog: 2


<h2>YOLO12n</h2>

In [17]:
model_yolo_12 = YOLO('yolo12n.pt')
cap = cv2.VideoCapture('test_2.mp4')
version = "YOLO12n"
calculate_inference_object(model_yolo_12, cap, version)

YOLO12n result : 
Processed 1081 frames in 21.87 seconds
YOLO12n - Average FPS (incl. display): 49.43
Average inference time per frame: 0.014 seconds

Detected objects summary:
bird: 27
airplane: 85
clock: 2
person: 2424
car: 254
motorcycle: 19
bus: 2
truck: 1
cell phone: 1
backpack: 18
teddy bear: 9
sports ball: 6
tie: 10
handbag: 21
suitcase: 8
carrot: 27
keyboard: 2
kite: 3
bed: 1
bench: 1
umbrella: 24
vase: 1
train: 9
dog: 1
boat: 3
